# 5X72 saved-input distance contrasts

This [prespecified diagnostic](5x72_contact_workflow.md) compares exact previously captured scoring inputs. **No new molecular input, preparation, score, search or optimization.** All eight matrices and every near-pair contrast are retained; both ROOT orders agree on the explicit source axis. Descriptive proximity does not define an interaction energy or prove a cause. Original versus generated H, omitted nonpolar H and partial PDBQT chemistry remain limits.

In [1]:
import gzip, json, sys
from pathlib import Path
checkout=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'devtools/qualify_5x72_contact.py').is_file())
sys.path.insert(0,str(checkout))
import numpy as np
import pandas as pd
import pyunitwizard as puw
import dockingmt as dmt
from devtools.qualify_5x72_contact import OUTPUT, SOURCE_AXIS, PROTEIN_COUNT, THRESHOLDS, load_inputs, sha
payload=OUTPUT.read_bytes();record=json.loads(gzip.decompress(payload))
assert sha(payload)=='84ad1212925bd0765f99b5e9a8932d5b7976e887f51a14964ec295b501aa5490'
assert sha(gzip.decompress(payload))=='f6765c5daa0c09923058c399b10e295f664aee266b1791558f9a4e6454425098'
assert sha(OUTPUT.with_name('producer_2026-10-09.py').read_bytes())==record['consumer_source_sha256']['devtools/qualify_5x72_contact.py']
assert record['new_searches']==record['new_fixed_evaluations']==0
load_inputs()  # Authenticate unchanged original rigid/reference/receptor archives.
entries=matrices=0
for ligand in record['ligands'].values():
    assert ligand['root_order_exact_on_source_axis'] and ligand['protein_prefix_exact']
    for arm in ligand['arms'].values():
        for context,item in arm.items():
            values=np.asarray(item['matrix_angstrom'])
            left=np.asarray([a['coordinates'] for a in item['canonical_ligand_atoms']])
            right=np.asarray([a['coordinates'] for a in item['receptor_atoms']])
            assert [a['source_atom_index'] for a in item['canonical_ligand_atoms']]==SOURCE_AXIS
            np.testing.assert_allclose(values,np.linalg.norm(left[:,None]-right[None,:],axis=2),rtol=0,atol=1e-10)
            assert all(dmt.verify_captured_inputs(a) for a in item['root_input_identities'].values())
            entries+=values.size;matrices+=1
        np.testing.assert_allclose(np.asarray(arm['occupied']['matrix_angstrom'])[:,:PROTEIN_COUNT],arm['sham']['matrix_angstrom'],rtol=0,atol=1e-10)
assert entries==298700 and matrices==8
print('Verified all 298,700 distances /8 matrices and captured-input digests; original archives unchanged.')

Verified all 298,700 distances /8 matrices and captured-input digests; original archives unchanged.


Complete heavy-pair populations and all four saved score contrasts follow. Protein matrices agree between sham and occupied. The large P69 score contrast already exists without its companion. Fixed distance thresholds remain descriptive; neither a pair count nor minimum distance assigns a score contribution.

In [2]:
summary=[];scores=[]
for name,ligand in record['ligands'].items():
    for arm,contexts in ligand['arms'].items():
        for context,item in contexts.items():
            for partition,pop in item['summary']['populations'].items():
                p=pop['heavy_heavy']
                summary.append({'ligand':name,'arm':arm,'context':context,'partition':partition,'heavy_pairs':p['pairs'],'minimum_angstrom':p['minimum_angstrom'],**{f'<{t:g} A':v for t,v in zip(THRESHOLDS,p['strict_threshold_counts'])}})
    for context in ['sham','occupied']:
        a=ligand['arms']['experimental_reference'][context]['saved_scores_kcal_mol']
        b=ligand['arms']['original_rigid'][context]['saved_scores_kcal_mol']
        scores.append({'ligand':name,'context':context,**{k+' rigid-minus-reference kcal/mol':b[k]-a[k] for k in a}})
display(pd.DataFrame(summary))
display(pd.DataFrame(scores))

,ligand,arm,context,partition,heavy_pairs,minimum_angstrom,<1.5 A,<2 A,<2.5 A,<3 A,<4 A,<5 A
0,p59,experimental_reference,occupied,companion,576,3.018935,0,0,0,0,10,45
1,p59,experimental_reference,occupied,protein,28968,2.537896,0,0,0,2,57,225
2,p59,experimental_reference,sham,companion,0,NaN,0,0,0,0,0,0
3,p59,experimental_reference,sham,protein,28968,2.537896,0,0,0,2,57,225
4,p59,original_rigid,occupied,companion,576,3.021996,0,0,0,0,16,49
5,p59,original_rigid,occupied,protein,28968,2.389969,0,0,2,3,47,234
6,p59,original_rigid,sham,companion,0,NaN,0,0,0,0,0,0
7,p59,original_rigid,sham,protein,28968,2.389969,0,0,2,3,47,234
8,p69,experimental_reference,occupied,companion,576,3.018935,0,0,0,0,10,45
9,p69,experimental_reference,occupied,protein,28968,2.796309,0,0,0,3,44,227


,ligand,context,fixed rigid-minus-reference kcal/mol,fixed.flex_inter rigid-minus-reference kcal/mol,fixed.flex_intra rigid-minus-reference kcal/mol,fixed.lig_inter rigid-minus-reference kcal/mol,fixed.lig_intra rigid-minus-reference kcal/mol,fixed.lig_intra_best_pose rigid-minus-reference kcal/mol,fixed.other_inter rigid-minus-reference kcal/mol,fixed.torsions rigid-minus-reference kcal/mol
0,p59,sham,2.043,0.0,0.0,2.281,0.521,0.521,0.0,-0.239
1,p59,occupied,1.983,0.0,0.0,2.215,0.521,0.521,0.0,-0.232
2,p69,sham,7.960,0.0,0.0,8.890,1.624,1.624,0.0,-0.930
3,p69,occupied,7.832,0.0,0.0,8.748,1.624,1.624,0.0,-0.916


Every original-rigid pair under 2.5 Å is shown below, including H classes; no outcome selects the inputs. P69's heavy protein pairs involve GLN116 and TYR149. P59 also has two heavy protein pairs below this threshold but a smaller score difference, demonstrating the limit of count-based interpretation. A future local displacement control would need a separate protocol; it is not performed here.

In [3]:
near=[];short=[]
for name,ligand in record['ligands'].items():
    for context,rows in ligand['near_pair_contrasts'].items():
        a=ligand['arms']['experimental_reference'][context];b=ligand['arms']['original_rigid'][context]
        left=np.asarray(a['matrix_angstrom']);right=np.asarray(b['matrix_angstrom'])
        expected=set(zip(*np.nonzero((left<5)|(right<5))))
        actual=set()
        for row in rows:
            i=SOURCE_AXIS.index(row['source_atom_index']);j=row['receptor_row'];actual.add((i,j))
            np.testing.assert_allclose([row['experimental_reference_angstrom'],row['original_rigid_angstrom'],row['rigid_minus_reference_angstrom']],[left[i,j],right[i,j],right[i,j]-left[i,j]],rtol=0,atol=1e-10)
            if row['original_rigid_angstrom']<2.5:
                latom=b['canonical_ligand_atoms'][i];ratom=b['receptor_atoms'][j]
                short.append({'ligand':name,'context':context,'source_index':row['source_atom_index'],'ligand_atom_type':latom['name']+'/'+latom['type'],'receptor':ratom['chain']+' '+ratom['group']+ratom['group_id']+' '+ratom['name']+'/'+ratom['type'],'partition':row['partition'],'pair_class':row['pair_class'],'reference_angstrom':row['experimental_reference_angstrom'],'rigid_angstrom':row['original_rigid_angstrom']})
        assert actual==expected
        near.append({'ligand':name,'context':context,'all_either_arm_below_5_A':len(rows)})
display(pd.DataFrame(near))
display(pd.DataFrame(short))
print('Distance associations only; no pair energy, causal attribution or production remedy established.')

,ligand,context,all_either_arm_below_5_A
0,p59,occupied,415
1,p59,sham,353
2,p69,occupied,356
3,p69,sham,300


,ligand,context,source_index,ligand_atom_type,receptor,partition,pair_class,reference_angstrom,rigid_angstrom
0,p59,occupied,15,C16/C,A GLN78 HE21/HD,protein,ligand_heavy_receptor_H,2.503981,2.074169
1,p59,occupied,16,O17/OA,A ARG61 NH1/N,protein,heavy_heavy,2.537896,2.389969
2,p59,occupied,16,O17/OA,A ARG61 HH12/HD,protein,ligand_heavy_receptor_H,1.793755,1.770042
3,p59,occupied,16,O17/OA,A GLN78 NE2/N,protein,heavy_heavy,2.950850,2.439326
4,p59,occupied,16,O17/OA,A GLN78 HE21/HD,protein,ligand_heavy_receptor_H,2.136099,1.608495
5,p59,sham,15,C16/C,A GLN78 HE21/HD,protein,ligand_heavy_receptor_H,2.503981,2.074169
6,p59,sham,16,O17/OA,A ARG61 NH1/N,protein,heavy_heavy,2.537896,2.389969
7,p59,sham,16,O17/OA,A ARG61 HH12/HD,protein,ligand_heavy_receptor_H,1.793755,1.770042
8,p59,sham,16,O17/OA,A GLN78 NE2/N,protein,heavy_heavy,2.950850,2.439326
9,p59,sham,16,O17/OA,A GLN78 HE21/HD,protein,ligand_heavy_receptor_H,2.136099,1.608495


Distance associations only; no pair energy, causal attribution or production remedy established.
